# 01. Khảo sát dữ liệu

Bộ dữ liệu Datathon 2026 Round 1 mô phỏng hoạt động của một doanh nghiệp bán lẻ thời trang. Ta sẽ xem 14 CSV trong `data/raw/` để hiểu mỗi bảng ghi gì, nối với bảng nào và có dấu hiệu nào cần tìm hiểu thêm.

Đọc từ trên xuống để nắm dữ liệu. Cuối notebook có chín câu hỏi dẫn đến [02. Kiểm chứng dữ liệu](02_kiem_chung_du_lieu.ipynb). Bảng kết quả được hiển thị ngay dưới các ô code.

## 1. Đọc 14 CSV

Mỗi bảng được đọc một lần. Các cột mã và ZIP được đọc dưới dạng chuỗi để giữ nguyên cách viết. Bảng bên dưới ghi rõ mỗi dòng đại diện cho gì và cột nào có thể dùng làm khóa.

In [29]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT") or Path.cwd()).expanduser().resolve()
root = next((path for path in (start, *start.parents) if (path / "data/raw").is_dir()), None)
if root is None:
    raise FileNotFoundError("Không tìm thấy data/raw/. Hãy mở thư mục gốc của repo.")

raw_dir = root / "data/raw"
table_specs = {
    "products": ("Một sản phẩm", ["product_id"]),
    "customers": ("Một khách hàng", ["customer_id"]),
    "promotions": ("Một chương trình khuyến mãi", ["promo_id"]),
    "geography": ("Một mã bưu chính", ["zip"]),
    "orders": ("Một đơn hàng", ["order_id"]),
    "order_items": ("Một dòng hàng trong đơn", []),
    "payments": ("Một bản ghi thanh toán", ["order_id"]),
    "shipments": ("Một bản ghi vận chuyển", ["order_id"]),
    "returns": ("Một bản ghi trả hàng", ["return_id"]),
    "reviews": ("Một bản ghi đánh giá", ["review_id"]),
    "sales": ("Doanh thu và giá vốn của một ngày", ["Date"]),
    "inventory": ("Một sản phẩm tại một kỳ tồn kho", ["product_id", "snapshot_date"]),
    "web_traffic": ("Một ngày truy cập web", ["date"]),
    "sample_submission": ("Một ngày cần dự báo", ["Date"]),
}
paths = {name: raw_dir / f"{name}.csv" for name in table_specs}
missing_files = [path.name for path in paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Thiếu CSV trong data/raw/: " + ", ".join(missing_files))

id_types = dict.fromkeys(
    ["customer_id", "order_id", "product_id", "zip", "promo_id", "promo_id_2", "return_id", "review_id"],
    "string",
)
data = {
    name: pd.read_csv(path, dtype=id_types, encoding="utf-8-sig", low_memory=False)
    for name, path in paths.items()
}

## 2. Quy mô, kiểu cột và ô thiếu

Bảng đầu cho biết quy mô từng CSV. Bảng tiếp theo ghi kiểu mà pandas đọc được và số ô thiếu theo cột. “Dòng trùng” đếm những dòng lặp lại sau lần xuất hiện đầu tiên, xét toàn bộ cột.

In [30]:
column_stats = []
table_stats = []
for name, frame in data.items():
    grain, key = table_specs[name]
    missing = frame.isna().sum()
    table_stats.append({
        "Bảng": name,
        "Một dòng ghi": grain,
        "Khóa khảo sát": ", ".join(key) if key else "Chưa có khóa riêng",
        "Số dòng": len(frame),
        "Số cột": len(frame.columns),
        "Ô thiếu": int(missing.sum()),
        "Dòng trùng": int(frame.duplicated().sum()),
    })
    column_stats.append(pd.DataFrame({
        "Bảng": name,
        "Cột": frame.columns,
        "Kiểu khi đọc": frame.dtypes.astype(str).to_numpy(),
        "Ô thiếu": missing.to_numpy(),
        "Tỷ lệ thiếu (%)": (missing / len(frame) * 100).round(2).to_numpy(),
    }))

overview = pd.DataFrame(table_stats).set_index("Bảng")
columns = pd.concat(column_stats, ignore_index=True)
display(overview)
display(columns.loc[columns["Ô thiếu"].gt(0)])

,Một dòng ghi,Khóa khảo sát,Số dòng,Số cột,Ô thiếu,Dòng trùng
Bảng,,,,,,
products,Một sản phẩm,product_id,2412,8,0,0
customers,Một khách hàng,customer_id,121930,7,0,0
promotions,Một chương trình khuyến mãi,promo_id,50,10,40,0
geography,Một mã bưu chính,zip,39948,4,0,0
orders,Một đơn hàng,order_id,646945,8,0,0
order_items,Một dòng hàng trong đơn,Chưa có khóa riêng,714669,7,1152816,0
payments,Một bản ghi thanh toán,order_id,646945,4,0,0
shipments,Một bản ghi vận chuyển,order_id,566067,4,0,0
returns,Một bản ghi trả hàng,return_id,39939,7,0,0


,Bảng,Cột,Kiểu khi đọc,Ô thiếu,Tỷ lệ thiếu (%)
21,promotions,applicable_category,object,40,80.00
42,order_items,promo_id,string,438353,61.34
43,order_items,promo_id_2,string,714463,99.97


### Xem cột và dữ liệu mẫu

Bên dưới là ba dòng đầu và danh sách cột của từng bảng. Với `sample_submission.csv`, ba cột `Date`, `Revenue` và `COGS` cho biết cấu trúc bài nộp. Mục V09 sẽ kiểm tra lịch dự báo chi tiết.

In [31]:
from IPython.display import Markdown

for name, frame in data.items():
    display(Markdown(f"**`{name}.csv`**"))
    display(columns.loc[columns["Bảng"].eq(name), ["Cột", "Kiểu khi đọc"]].reset_index(drop=True))
    display(frame.head(3))

**`products.csv`**

,Cột,Kiểu khi đọc
0,product_id,string
1,product_name,object
2,category,object
3,segment,object
4,size,object
5,color,object
6,price,float64
7,cogs,float64


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278


**`customers.csv`**

,Cột,Kiểu khi đọc
0,customer_id,string
1,zip,string
2,city,object
3,signup_date,object
4,gender,object
5,age_group,object
6,acquisition_channel,object


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search


... (32 outputs left out)

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct


**`sample_submission.csv`**

,Cột,Kiểu khi đọc
0,Date,object
1,Revenue,float64
2,COGS,float64


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12


### Các cột số chính có giá trị trong khoảng nào?

Bảng dưới giải thích ngắn ý nghĩa của từng cột được chọn, rồi cho biết số giá trị số hợp lệ, min, trung bình, trung vị và max. Trung vị là giá trị ở giữa khi sắp xếp các số, nên giúp đọc mức điển hình khi có giá trị rất lớn hoặc rất nhỏ.

Thống kê bỏ qua ô trống, giá trị không đọc được thành số và vô cực. Các khoản tiền giữ đơn vị của nguồn. Mục V02 trong notebook kiểm chứng sẽ đối soát tiền chính xác bằng `Decimal`.

In [32]:
numeric_fields = {
    "products": {
        "price": "Giá sản phẩm ghi trong danh mục",
        "cogs": "Giá vốn tham chiếu của một sản phẩm",
    },
    "order_items": {
        "quantity": "Số sản phẩm trên một dòng hàng",
        "unit_price": "Đơn giá ghi trên dòng hàng",
        "discount_amount": "Khoản giảm giá ghi cho cả dòng hàng",
    },
    "payments": {
        "payment_value": "Số tiền ghi trên bản ghi thanh toán",
    },
    "returns": {
        "return_quantity": "Số sản phẩm trả trên bản ghi",
        "refund_amount": "Số tiền hoàn ghi trên bản ghi trả hàng",
    },
    "inventory": {
        "stock_on_hand": "Số lượng đang tồn tại kỳ quan sát",
        "units_received": "Số lượng nhập ghi nhận trong kỳ",
        "units_sold": "Số lượng bán ghi nhận trong tồn kho",
        "stockout_days": "Số ngày thiếu hàng do nguồn ghi nhận",
        "days_of_supply": "Số ngày đủ hàng do nguồn ghi nhận",
        "fill_rate": "Chỉ số đáp ứng hàng do nguồn ghi nhận",
        "sell_through_rate": "Tỷ lệ bán ra do nguồn ghi nhận",
    },
    "sales": {
        "Revenue": "Doanh thu gộp trước giảm giá theo ngày",
        "COGS": "Giá vốn ghi theo ngày trong bảng Sales",
    },
    "web_traffic": {
        "sessions": "Số phiên truy cập ghi nhận trong ngày",
        "bounce_rate": "Chỉ số thoát do nguồn ghi nhận, đơn vị cần xác minh",
    },
}

numeric_rows = []
for table_name, fields in numeric_fields.items():
    for column_name, meaning in fields.items():
        values = pd.to_numeric(data[table_name][column_name], errors="coerce")
        values = values.replace([float("inf"), -float("inf")], float("nan"))
        stats = values.agg(["count", "min", "mean", "median", "max"])
        numeric_rows.append({
            "Cột": f"{table_name}.{column_name}",
            "Ý nghĩa": meaning,
            "Số giá trị hợp lệ": int(stats["count"]),
            "Min": stats["min"],
            "Trung bình": stats["mean"],
            "Trung vị": stats["median"],
            "Max": stats["max"],
        })

numeric_summary = pd.DataFrame(numeric_rows)
with pd.option_context("display.float_format", lambda v: f"{v:,.6f}".rstrip("0").rstrip(".")):
    display(numeric_summary)

,Cột,Ý nghĩa,Số giá trị hợp lệ,Min,Trung bình,Trung vị,Max
0,products.price,Giá sản phẩm ghi trong danh mục,2412,9.056594,"4,928.216231","4,399.605","40,950"
1,products.cogs,Giá vốn tham chiếu của một sản phẩm,2412,5.183829,"3,868.346732","3,184.934093","38,902.5"
2,order_items.quantity,Số sản phẩm trên một dòng hàng,714669,1,4.495988,4,8
3,order_items.unit_price,Đơn giá ghi trên dòng hàng,714669,392.57,"5,114.690157","4,257.77","43,056"
4,order_items.discount_amount,Khoản giảm giá ghi cho cả dòng hàng,714669,0,"1,048.887415",0,"35,235.47"
5,payments.payment_value,Số tiền ghi trên bản ghi thanh toán,646945,389.74,"24,238.334426","17,229.44","331,570.4"
6,returns.return_quantity,Số sản phẩm trả trên bản ghi,39939,1,2.743834,2,8
7,returns.refund_amount,Số tiền hoàn ghi trên bản ghi trả hàng,39939,458.81,"12,784.458964","7,888.88","160,937.94"
8,inventory.stock_on_hand,Số lượng đang tồn tại kỳ quan sát,60247,3,189.298455,62,"2,673"
9,inventory.units_received,Số lượng nhập ghi nhận trong kỳ,60247,1,18.046807,6,817


## 3. Khóa và liên kết cơ bản

Một khóa có thể nhận diện dòng khi đủ giá trị và duy nhất. `order_items.csv` chưa có mã riêng cho từng dòng, nên ta đếm riêng các cặp đơn–sản phẩm lặp. Một cặp có thể xuất hiện nhiều lần trong cùng đơn.

Với khóa liên kết, tách hai trường hợp “ô trống” và “có mã nhưng không thấy trong bảng đích”. Hai ô mã khuyến mãi có thể để trống.

In [33]:
key_stats = []
for name, (_, key) in table_specs.items():
    if key:
        frame = data[name]
        key_stats.append({
            "Bảng": name,
            "Cột khóa": ", ".join(key),
            "Dòng thiếu khóa": int(frame[key].isna().any(axis=1).sum()),
            "Dòng thuộc nhóm khóa trùng": int(frame.duplicated(key, keep=False).sum()),
        })
display(pd.DataFrame(key_stats))

items = data["order_items"]
pair_counts = items.groupby(["order_id", "product_id"], dropna=False, sort=False).size()
repeated_pair_counts = pair_counts.loc[pair_counts.gt(1)]
display(pd.DataFrame([{
    "Bảng": "order_items",
    "Cặp đơn–sản phẩm lặp": len(repeated_pair_counts),
    "Dòng thuộc các cặp lặp": int(repeated_pair_counts.sum()),
}]))

links = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "zip", "geography", "zip"),
    ("customers", "zip", "geography", "zip"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),
    ("returns", "order_id", "orders", "order_id"),
    ("returns", "product_id", "products", "product_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "product_id", "products", "product_id"),
    ("reviews", "customer_id", "customers", "customer_id"),
    ("inventory", "product_id", "products", "product_id"),
    ("order_items", "promo_id", "promotions", "promo_id"),
    ("order_items", "promo_id_2", "promotions", "promo_id"),
]
link_stats = []
for source, field, target, target_key in links:
    values = data[source][field]
    present = values.dropna()
    link_stats.append({
        "Cột nguồn": f"{source}.{field}",
        "Cột đích": f"{target}.{target_key}",
        "Ô trống": len(values) - len(present),
        "Mã không thấy ở đích": int((~present.isin(data[target][target_key])).sum()),
    })
display(pd.DataFrame(link_stats))

,Bảng,Cột khóa,Dòng thiếu khóa,Dòng thuộc nhóm khóa trùng
0,products,product_id,0,0
1,customers,customer_id,0,0
2,promotions,promo_id,0,0
3,geography,zip,0,0
4,orders,order_id,0,0
5,payments,order_id,0,0
6,shipments,order_id,0,0
7,returns,return_id,0,0
8,reviews,review_id,0,0
9,sales,Date,0,0


,Bảng,Cặp đơn–sản phẩm lặp,Dòng thuộc các cặp lặp
0,order_items,16,32


,Cột nguồn,Cột đích,Ô trống,Mã không thấy ở đích
0,orders.customer_id,customers.customer_id,0,0
1,orders.zip,geography.zip,0,0
2,customers.zip,geography.zip,0,0
3,order_items.order_id,orders.order_id,0,0
4,order_items.product_id,products.product_id,0,0
5,payments.order_id,orders.order_id,0,0
6,shipments.order_id,orders.order_id,0,0
7,returns.order_id,orders.order_id,0,0
8,returns.product_id,products.product_id,0,0
9,reviews.order_id,orders.order_id,0,0


## 4. Phạm vi thời gian

Xem ngày đầu, ngày cuối và số ngày hoặc tháng có quan sát. Ngày đăng ký của khách hàng dùng định dạng tháng/ngày/năm. Các cột ngày còn lại dùng năm-tháng-ngày.

In [34]:
date_columns = {
    "customers": ["signup_date"],
    "promotions": ["start_date", "end_date"],
    "orders": ["order_date"],
    "shipments": ["ship_date", "delivery_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "inventory": ["snapshot_date"],
    "web_traffic": ["date"],
}

date_stats = []
for name, fields in date_columns.items():
    for field in fields:
        values = data[name][field]
        date_format = "%Y-%m-%d"
        dates = pd.to_datetime(values, format=date_format, errors="coerce")
        date_stats.append({
            "Cột ngày": f"{name}.{field}",
            "Ngày đầu": dates.min().date(),
            "Ngày cuối": dates.max().date(),
            "Số ngày": dates.nunique(),
            "Số tháng": dates.dt.to_period("M").nunique(),
            "Ô trống": int(values.isna().sum()),
            "Ngày không đọc được": int((values.notna() & dates.isna()).sum()),
        })
display(pd.DataFrame(date_stats))

,Cột ngày,Ngày đầu,Ngày cuối,Số ngày,Số tháng,Ô trống,Ngày không đọc được
0,customers.signup_date,2012-01-17,2022-12-31,3941,132,0,0
1,promotions.start_date,2013-01-31,2022-11-18,50,50,0,0
2,promotions.end_date,2013-03-01,2022-12-31,50,50,0,0
3,orders.order_date,2012-07-04,2022-12-31,3833,126,0,0
4,shipments.ship_date,2012-07-04,2022-12-29,3831,126,0,0
5,shipments.delivery_date,2012-07-06,2022-12-31,3831,126,0,0
6,returns.return_date,2012-07-11,2022-12-31,3806,126,0,0
7,reviews.review_date,2012-07-10,2022-12-31,3825,126,0,0
8,sales.Date,2012-07-04,2022-12-31,3833,126,0,0
9,sample_submission.Date,2023-01-01,2024-07-01,548,19,0,0


## 5. Độ phủ đơn hàng, khách hàng và sản phẩm

Đếm trạng thái ghi trên đơn, số dòng hàng mỗi đơn và mã xuất hiện trong từng bảng. “Có đơn hàng” ở đây tính trên mọi trạng thái. Độ phủ tồn kho tính theo mã sản phẩm có ít nhất một kỳ được ghi nhận.

In [35]:
orders = data["orders"]
status = orders["order_status"].value_counts(dropna=False).rename("Số đơn").to_frame()
status["Tỷ lệ (%)"] = (status["Số đơn"] / len(orders) * 100).round(2)
display(status.rename_axis("Trạng thái"))

items_per_order = items["order_id"].value_counts(sort=False)
lines_per_order = orders["order_id"].map(items_per_order).fillna(0).astype(int)
order_sizes = lines_per_order.value_counts().sort_index().reindex(
    range(lines_per_order.max() + 1), fill_value=0
)
display(order_sizes.rename("Số đơn").rename_axis("Số dòng hàng mỗi đơn").to_frame())

customer_ids = set(data["customers"]["customer_id"].dropna())
ordered_customer_ids = set(orders["customer_id"].dropna())
product_ids = set(data["products"]["product_id"].dropna())
ordered_product_ids = set(items["product_id"].dropna())
inventory_product_ids = set(data["inventory"]["product_id"].dropna())

coverage = {
    "Khách trong danh mục": len(customer_ids),
    "Khách trong danh mục có đơn": len(customer_ids & ordered_customer_ids),
    "Khách trong danh mục chưa có đơn": len(customer_ids - ordered_customer_ids),
    "Mã khách trên đơn vắng trong danh mục": len(ordered_customer_ids - customer_ids),
    "Sản phẩm trong danh mục": len(product_ids),
    "Sản phẩm xuất hiện trong dòng hàng": len(ordered_product_ids),
    "Sản phẩm xuất hiện trong tồn kho": len(inventory_product_ids),
    "Sản phẩm trong danh mục vắng trong dòng hàng": len(product_ids - ordered_product_ids),
    "Sản phẩm trong danh mục vắng trong tồn kho": len(product_ids - inventory_product_ids),
    "Sản phẩm có dòng hàng vắng trong tồn kho": len(ordered_product_ids - inventory_product_ids),
    "Sản phẩm có tồn kho vắng trong dòng hàng": len(inventory_product_ids - ordered_product_ids),
}
display(pd.Series(coverage, name="Số mã").rename_axis("Độ phủ").to_frame())

,Số đơn,Tỷ lệ (%)
Trạng thái,,
delivered,516716,79.87
cancelled,59462,9.19
returned,36142,5.59
shipped,13773,2.13
paid,13577,2.10
created,7275,1.12


,Số đơn
Số dòng hàng mỗi đơn,
0,0
1,580488
2,65256
3,1137
4,62
5,2


,Số mã
Độ phủ,
Khách trong danh mục,121930
Khách trong danh mục có đơn,90246
Khách trong danh mục chưa có đơn,31684
Mã khách trên đơn vắng trong danh mục,0
Sản phẩm trong danh mục,2412
Sản phẩm xuất hiện trong dòng hàng,1598
Sản phẩm xuất hiện trong tồn kho,1624
Sản phẩm trong danh mục vắng trong dòng hàng,814
Sản phẩm trong danh mục vắng trong tồn kho,788


## 6. Câu hỏi cần kiểm chứng

Các câu hỏi dưới đây xuất phát từ những cột và bảng vừa xem. Mở [02. Kiểm chứng dữ liệu](02_kiem_chung_du_lieu.ipynb) và tìm theo mã ở cột đầu.

| Mục | Quan sát từ khảo sát | Câu hỏi |
|---|---|---|
| V01 | Có ngày đặt đơn và ngày đăng ký khách. | Có đơn nào ghi ngày đặt trước ngày đăng ký của khách? |
| V02 | Dòng hàng ghi số lượng, đơn giá và giảm giá. Các bảng khác ghi thanh toán, giá vốn và Revenue. | Tổng tiền sau giảm giá khớp thanh toán đến mức nào? Revenue và COGS trong Sales khớp phép tính nào? |
| V03 | Đơn hàng có trạng thái `returned`. Bảng Returns ghi số lượng trả và tiền hoàn. | Các đơn `returned` có chi tiết trả hàng tương ứng không? Có đơn nào hoàn tiền vượt thanh toán? |
| V04 | Tồn kho ghi `units_sold` theo sản phẩm và kỳ, còn giao dịch có `quantity`. | Hai nguồn có cùng phạm vi sản phẩm–tháng và lượng bán có khớp không? Kết quả thay đổi thế nào khi chỉ xét đơn `delivered`? |
| V05 | Bảng tồn kho có `reorder_flag` cùng các cờ thiếu và dư hàng. | `reorder_flag` có cả hai giá trị 0 và 1 để phân biệt hai lớp không? |
| V06 | Các cột `promo_id`, `promo_id_2` và `applicable_category` có ô trống. | Có bao nhiêu dòng dùng đồng thời hai mã? Tình trạng có mã liên hệ thế nào với giảm giá? |
| V07 | Bảng truy cập web có trường số `bounce_rate`. | Giá trị nằm trong khoảng nào? Có đủ thông tin để xác định cách đọc đơn vị không? |
| V08 | Bảng web có `traffic_source`, còn bảng khách có `acquisition_channel`. | Nhãn nguồn web xuất hiện thế nào theo ngày? Hai cột giống nhau ở điểm nào và có thể suy ra gì từ chúng? |
| V09 | Sales và mẫu nộp đều có `Date`, `Revenue` và `COGS`. | Lịch mẫu có liên tục, có ngày trùng hoặc giao với lịch sử Sales không? |

Các kiểm tra riêng về vận chuyển và phương trình tồn kho giữa hai kỳ được trình bày ở bước tiền xử lý và các notebook nghiệp vụ liên quan.